In [1]:
#!pip install datasets pypdf


In [5]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"


In [9]:
# Import necessary libraries
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from pypdf import PdfReader
import re

# Load a pre-trained model for question generation
question_generation_model = "valhalla/t5-small-qg-prepend"
#tokenizer = AutoTokenizer.from_pretrained(question_generation_model)
tokenizer = AutoTokenizer.from_pretrained(question_generation_model, use_fast=False)

model = AutoModelForSeq2SeqLM.from_pretrained(question_generation_model)

# Initialize the question-generation pipeline
qg_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    num_beams=3  # Enable beam search
)
# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

# Function to clean and preprocess text
def clean_text(text):
    # Remove excessive whitespace and special characters
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9.,!?;'\s]", "", text)
    return text.strip()

# Function to split text into meaningful chunks
def split_text_into_chunks(text, max_tokens=200):
    sentences = re.split(r'(?<=[.!?]) +', text)  # Split by sentence boundaries
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        # Token-level splitting
        if len(current_chunk.split()) + len(sentence.split()) <= max_tokens:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Function to generate QA pairs from text
def extract_qa_pairs_from_text(text):
    qa_pairs = []
    chunks = split_text_into_chunks(text)
    for chunk in chunks:
        input_text = f"generate questions: {chunk}"
        questions = qg_pipeline(input_text, max_length=64, num_return_sequences=3)
        for question in questions:
            qa_pairs.append({"question": question['generated_text'], "context": chunk})
    return qa_pairs

# Main pipeline for PDF
def extract_qa_pairs_from_pdf(pdf_path):
    # Step 1: Extract and clean text
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_text(raw_text)

    # Step 2: Generate QA pairs
    qa_pairs = extract_qa_pairs_from_text(cleaned_text)

    return qa_pairs

# Example Usage
pdf_path = "example.pdf"  # Replace with your PDF file path

print("\n--- Extracting QA pairs from PDF ---")
qa_pairs = extract_qa_pairs_from_pdf(pdf_path)

# Print the extracted QA pairs
for i, qa in enumerate(qa_pairs):
    print(f"Q{i+1}: {qa['question']}")
    print(f"A{i+1}: {qa['context']}")
    print()


pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]


--- Extracting QA pairs from PDF ---


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q1: What is the name of the source of energy for Canada's Future A Clean Electricity Strategy 155197 minutes?
A1: naturalresources.canada.ca ournaturalresourcesenergysourcesdistributionelectricityinfrastru Powering Canadas Future A Clean Electricity Strategy 155197 minutes Table of Contents Foreword  Clean Electricity Strategy 1.0 The Case for Clean Electricity 1.1 Laying Out a Clean Electricity Strategy for Canada 1.2 A Strategy informed by extensive engagement, electricity sector experts, and Indigenous energy leaders 1.3 Key Guiding Principles 2.0 Toward the Grid of the Future 2.1 Global Context 2.2 Canadian Context 2.3 Regional Context 3.0 Federal Action 3.1 Focus Area 1 Growing the Grid and Managing Demand 3.2 Focus Area 2 Providing Policy Certainty and Smoothing the Path 3.3 Focus Area 3 Collaborating on Tailored Approaches for Every Region 4.0 Next Steps Annex 1  Canada Electricity Advisory Council Recommendations Annex 2  Wahilatoos Indigenous Council Recommendations List of Fi